In [5]:
import time
import pandas as pd

STATES = ["AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID",
          "IL","IN","IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO",
          "MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA",
          "RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY"]

URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={}"

def pull(series_id, col):
    df = pd.read_csv(URL.format(series_id))             #read FRED data
    df.columns = ["report_date", col]                      
    df["report_date"] = pd.to_datetime(df["report_date"])
    df[col] = pd.to_numeric(df[col], errors="coerce")      
    return df

frames = []
for st in STATES:
    ur  = pull(f"{st}UR", "unemployment_rate")             #Unemployment Pull
    hpi = pull(f"{st}STHPI", "hpi_index")                  #HPI Pull
    df = ur.merge(hpi, on="report_date", how="left")
    df["hpi_index"] = df["hpi_index"].ffill()              # Fill HPI quarters into foward months
    df["state"] = st
    frames.append(df)
    time.sleep(0.5)                                       

econ = pd.concat(frames, ignore_index=True)
econ["mortgage_rate_30yr"] = None                          
econ = econ[["report_date","state","unemployment_rate","hpi_index","mortgage_rate_30yr"]]

print(econ.shape)


(31008, 5)


In [6]:
print(econ.head())

  report_date state  unemployment_rate  hpi_index mortgage_rate_30yr
0  1976-01-01    AL                6.7      76.85               None
1  1976-02-01    AL                6.6      76.85               None
2  1976-03-01    AL                6.6      76.85               None
3  1976-04-01    AL                6.5      73.94               None
4  1976-05-01    AL                6.4      73.94               None


In [7]:
import sys
sys.path.append("..")  # db_config.py lives in the repo root
from db_config import get_connection, df_rows, insert_rows



conn = get_connection() 
cursor = conn.cursor()
cursor.execute("""DELETE FROM economic_indicators""")

try:
    insert_rows(conn, "economic_indicators", df_rows(econ))
    conn.commit()
    print(f"{len(econ)} rows inserted")
except Exception as e:
    conn.rollback()
    print(f"Insert failed: {e}")